# 🚚 Ödev: Custom Chat Template + Tool-Calling Asistan

**Senaryo:** Kargo sevkiyat ve takip sistemi
**Veritabanı:** SQLite (okuma + yazma)
**Model:** gpt-4o-mini (function calling)

### Teslim edilecekler
| # | Ödev | Çıktı |
|---|------|-------|
| 1 | Custom Chat Template | `chat_template.jinja` |
| 2 | Tool-Calling Asistan | GitHub repo + HF Space |

### Notebook akışı
1. Kurulum
2. `chat_template.jinja` yaz ve **render testi** yap
3. `database.py` — SQLite şeması, CRUD
4. `tools.py` — 4 araç + JSON şemaları
5. `agent.py` — tool calling döngüsü
6. Uçtan uca testler (halüsinasyon, kapasite, iş kuralları)
7. `app.py` — Gradio arayüzü, Colab'da dene
8. Hugging Face Space'e yayınla

---


## 0. Kurulum

In [1]:
%%capture
!pip install -q gradio openai jinja2 huggingface_hub

In [3]:
import os, json, sqlite3
from pathlib import Path
from getpass import getpass


def secret_al(ad: str) -> str:
    """Colab Secrets'tan okur; başarısız olursa gizli girdi ister."""
    try:
        from google.colab import userdata
        deger = userdata.get(ad)
        if deger:
            print(f"  ✓ {ad} — Colab Secrets'tan okundu")
            return deger
        print(f"  ⚠ {ad} — Secrets'ta boş görünüyor")
    except ImportError:
        print(f"  · {ad} — Colab dışı ortam")
    except Exception as e:
        # SecretNotFoundError / NotebookAccessError vb.
        print(f"  ⚠ {ad} — okunamadı ({type(e).__name__})")
        print(f"     Sol menü 🔑 → '{ad}' var mı ve 'Not defteri erişimi' açık mı kontrol edin")

    return getpass(f"     {ad} değerini yapıştırın: ").strip()


print("Anahtarlar okunuyor...")
OPENAI_KEY = secret_al("OPENAI_API_KEY")
HF_TOKEN   = secret_al("HF_TOKEN")

os.environ["OPENAI_API_KEY"] = OPENAI_KEY
os.environ["HF_TOKEN"] = HF_TOKEN

HF_USERNAME = "cihatyldz"                        # ← kendi kullanıcı adınız
SPACE_NAME  = "lojistik-kargo-asistani"
SPACE_ID    = f"{HF_USERNAME}/{SPACE_NAME}"

# Doğrulama — değerleri asla tam basma
print(f"\nOpenAI key : {OPENAI_KEY[:7]}...{OPENAI_KEY[-4:]}  ({len(OPENAI_KEY)} karakter)")
print(f"HF token   : {HF_TOKEN[:6]}...{HF_TOKEN[-4:]}  ({len(HF_TOKEN)} karakter)")
print(f"Space      : {SPACE_ID}")
print("✓ Kurulum tamam")

Anahtarlar okunuyor...
  ✓ OPENAI_API_KEY — Colab Secrets'tan okundu
  ✓ HF_TOKEN — Colab Secrets'tan okundu

OpenAI key : sk-proj...1E0A  (164 karakter)
HF token   : hf_XAb...ZlcZ  (37 karakter)
Space      : cihatyldz/lojistik-kargo-asistani
✓ Kurulum tamam


In [4]:
import os, json, sqlite3
from pathlib import Path

try:
    from google.colab import userdata
    OPENAI_KEY = userdata.get('OPENAI_API_KEY')
    HF_TOKEN   = userdata.get('HF_TOKEN')
except Exception:
    OPENAI_KEY = input("OpenAI API key: ")
    HF_TOKEN   = input("HuggingFace token: ")

os.environ["OPENAI_API_KEY"] = OPENAI_KEY
os.environ["HF_TOKEN"] = HF_TOKEN

HF_USERNAME = "cihatyldz"                        # ← kendi kullanıcı adınız
SPACE_NAME  = "lojistik-kargo-asistani"
SPACE_ID    = f"{HF_USERNAME}/{SPACE_NAME}"

print(f"Space: {SPACE_ID}")
print("✓ Kurulum tamam")

Space: cihatyldz/lojistik-kargo-asistani
✓ Kurulum tamam


---
# Ödev 1: Custom Chat Template (Jinja2)

Dört rolü ayrı ayrı sarmalayan, tool calling destekli şablon.

| Rol | Sarmalama |
|-----|-----------|
| `system` | `<\|im_start\|>system ... <\|im_end\|>` (araç şemaları içeri gömülür) |
| `user` | `<\|im_start\|>user ... <\|im_end\|>` |
| `assistant` | `<\|im_start\|>assistant ... <\|tool_call\|>{...}<\|/tool_call\|> ... <\|im_end\|>` |
| `tool` | `<\|im_start\|>tool <\|tool_response\|>{...}<\|/tool_response\|><\|im_end\|>` |


In [5]:
template_kodu = r'''{#-
  LojistikAI — Özel Chat Template (Jinja2)
  ========================================
  Rol sarmalama:
    system    → <|im_start|>system ... <|im_end|>
    user      → <|im_start|>user ... <|im_end|>
    assistant → <|im_start|>assistant ... <|im_end|>
    tool      → <|im_start|>tool ... <|im_end|>

  Tool calling:
    - Araç tanımları (tools) sistem mesajının içine JSON şeması olarak gömülür
    - Model araç çağırırken  <|tool_call|>{"name": ..., "arguments": {...}}<|/tool_call|>
    - Araç sonucu           <|tool_response|>{...}<|/tool_response|>

  Desteklenen değişkenler:
    messages              — sohbet geçmişi (zorunlu)
    tools                 — araç tanımları listesi (opsiyonel)
    add_generation_prompt — üretim için asistan başlığı aç (opsiyonel)
    enable_thinking       — düşünme bloğu aç (opsiyonel)

  Yazar: Cihat Yıldız
-#}

{#- ===== 1. Sistem mesajı ve araç tanımları ===== -#}
{%- set ns = namespace(system_content='', loop_messages=messages) -%}

{%- if messages[0]['role'] == 'system' -%}
    {%- set ns.system_content = messages[0]['content'] | trim -%}
    {%- set ns.loop_messages = messages[1:] -%}
{%- else -%}
    {%- set ns.system_content = 'Sen LojistikAI\'sın — kargo ve sevkiyat işlemlerinde yardımcı olan bir asistansın.' -%}
{%- endif -%}

{{- '<|im_start|>system\n' -}}
{{- ns.system_content -}}

{%- if tools is defined and tools -%}
    {{- '\n\n# Kullanılabilir Araçlar\n\n' -}}
    {{- 'Aşağıdaki araçları kullanabilirsin. Bir aracı çağırmak için şu formatı kullan:\n' -}}
    {{- '<|tool_call|>{"name": "araç_adı", "arguments": {"parametre": "değer"}}<|/tool_call|>\n\n' -}}
    {{- 'Araç tanımları (JSON Schema):\n' -}}
    {%- for tool in tools -%}
        {%- if tool.function is defined -%}
            {{- tool.function | tojson -}}
        {%- else -%}
            {{- tool | tojson -}}
        {%- endif -%}
        {{- '\n' -}}
    {%- endfor -%}
    {{- '\nÖnemli: Yanıtlarını yalnızca araçlardan dönen gerçek veriye dayandır. ' -}}
    {{- 'Veritabanında olmayan bir bilgiyi varmış gibi sunma.' -}}
{%- endif -%}
{{- '<|im_end|>\n' -}}

{#- ===== 2. Sohbet döngüsü ===== -#}
{%- for message in ns.loop_messages -%}

    {#- --- Kullanıcı mesajı --- -#}
    {%- if message['role'] == 'user' -%}
        {{- '<|im_start|>user\n' -}}
        {%- if message['content'] is string -%}
            {{- message['content'] | trim -}}
        {%- else -%}
            {%- for item in message['content'] -%}
                {%- if item['type'] == 'text' -%}
                    {{- item['text'] | trim -}}
                {%- elif item['type'] == 'image' -%}
                    {{- '<|image|>' -}}
                {%- endif -%}
            {%- endfor -%}
        {%- endif -%}
        {{- '<|im_end|>\n' -}}

    {#- --- Asistan mesajı (metin ve/veya araç çağrısı) --- -#}
    {%- elif message['role'] == 'assistant' -%}
        {{- '<|im_start|>assistant\n' -}}

        {%- if message['content'] -%}
            {{- message['content'] | trim -}}
        {%- endif -%}

        {%- if message['tool_calls'] is defined and message['tool_calls'] -%}
            {%- for tc in message['tool_calls'] -%}
                {%- set fn = tc['function'] if tc['function'] is defined else tc -%}
                {{- '\n<|tool_call|>' -}}
                {{- '{"name": "' + fn['name'] + '", "arguments": ' -}}
                {%- if fn['arguments'] is string -%}
                    {{- fn['arguments'] -}}
                {%- else -%}
                    {{- fn['arguments'] | tojson -}}
                {%- endif -%}
                {{- '}<|/tool_call|>' -}}
            {%- endfor -%}
        {%- endif -%}
        {{- '<|im_end|>\n' -}}

    {#- --- Araç sonucu --- -#}
    {%- elif message['role'] == 'tool' -%}
        {{- '<|im_start|>tool\n' -}}
        {%- if message['name'] is defined -%}
            {{- '[' + message['name'] + '] ' -}}
        {%- endif -%}
        {{- '<|tool_response|>' -}}
        {%- if message['content'] is string -%}
            {{- message['content'] -}}
        {%- else -%}
            {{- message['content'] | tojson -}}
        {%- endif -%}
        {{- '<|/tool_response|><|im_end|>\n' -}}

    {#- --- Bilinmeyen rol --- -#}
    {%- else -%}
        {{- raise_exception('Desteklenmeyen rol: ' + message['role'] +
            '. Geçerli roller: system, user, assistant, tool') -}}
    {%- endif -%}

{%- endfor -%}

{#- ===== 3. Üretim başlığı ===== -#}
{%- if add_generation_prompt -%}
    {{- '<|im_start|>assistant\n' -}}
    {%- if enable_thinking is defined and enable_thinking -%}
        {{- '<|think|>\n' -}}
    {%- endif -%}
{%- endif -%}
'''

with open("chat_template.jinja", "w", encoding="utf-8") as f:
    f.write(template_kodu)

print(f"✓ chat_template.jinja yazıldı ({len(template_kodu):,} karakter)")

✓ chat_template.jinja yazıldı (4,641 karakter)


### 1.1 Şablonu Test Et

In [6]:
from jinja2 import Environment, BaseLoader

def raise_exception(msg):
    raise ValueError(msg)

env = Environment(loader=BaseLoader())
env.globals["raise_exception"] = raise_exception
env.policies["json.dumps_kwargs"] = {"ensure_ascii": False}
tpl = env.from_string(template_kodu)

# Örnek araç tanımı
ornek_tools = [{
    "type": "function",
    "function": {
        "name": "get_services",
        "description": "Kargo hizmetlerini listeler",
        "parameters": {"type": "object",
                       "properties": {"kategori": {"type": "string"}},
                       "required": []},
    },
}]

ornek_mesajlar = [
    {"role": "system", "content": "Sen LojistikAI'sın."},
    {"role": "user", "content": "Özel hizmetleri göster"},
    {"role": "assistant", "content": None, "tool_calls": [
        {"id": "c1", "type": "function",
         "function": {"name": "get_services", "arguments": {"kategori": "ozel"}}}
    ]},
    {"role": "tool", "name": "get_services",
     "content": '{"hizmet_sayisi": 2, "hizmetler": [{"kod": "SGK"}, {"kod": "KRM"}]}'},
    {"role": "assistant", "content": "Soğuk zincir ve kırılabilir eşya hizmetlerimiz var."},
    {"role": "user", "content": "Sevkiyat oluşturmak istiyorum"},
]

cikti = tpl.render(messages=ornek_mesajlar, tools=ornek_tools, add_generation_prompt=True)
print(cikti)

<|im_start|>system
Sen LojistikAI'sın.

# Kullanılabilir Araçlar

Aşağıdaki araçları kullanabilirsin. Bir aracı çağırmak için şu formatı kullan:
<|tool_call|>{"name": "araç_adı", "arguments": {"parametre": "değer"}}<|/tool_call|>

Araç tanımları (JSON Schema):
{"name": "get_services", "description": "Kargo hizmetlerini listeler", "parameters": {"type": "object", "properties": {"kategori": {"type": "string"}}, "required": []}}

Önemli: Yanıtlarını yalnızca araçlardan dönen gerçek veriye dayandır. Veritabanında olmayan bir bilgiyi varmış gibi sunma.<|im_end|>
<|im_start|>user
Özel hizmetleri göster<|im_end|>
<|im_start|>assistant

<|tool_call|>{"name": "get_services", "arguments": {"kategori": "ozel"}}<|/tool_call|><|im_end|>
<|im_start|>tool
[get_services] <|tool_response|>{"hizmet_sayisi": 2, "hizmetler": [{"kod": "SGK"}, {"kod": "KRM"}]}<|/tool_response|><|im_end|>
<|im_start|>assistant
Soğuk zincir ve kırılabilir eşya hizmetlerimiz var.<|im_end|>
<|im_start|>user
Sevkiyat oluşturmak 

In [7]:
# Hata yönetimi testi — geçersiz rol
try:
    tpl.render(messages=[{"role": "hacker", "content": "x"}], add_generation_prompt=False)
    print("✗ Geçersiz rol yakalanmadı!")
except ValueError as e:
    print(f"✓ Geçersiz rol yakalandı:\n  {e}")

# Sistem mesajı olmadan da çalışmalı
cikti2 = tpl.render(
    messages=[{"role": "user", "content": "Merhaba"}],
    add_generation_prompt=True,
)
print(f"\n✓ Sistem mesajsız render:\n{cikti2}")

✓ Geçersiz rol yakalandı:
  Desteklenmeyen rol: hacker. Geçerli roller: system, user, assistant, tool

✓ Sistem mesajsız render:
<|im_start|>system
Sen LojistikAI'sın — kargo ve sevkiyat işlemlerinde yardımcı olan bir asistansın.<|im_end|>
<|im_start|>user
Merhaba<|im_end|>
<|im_start|>assistant



---
# Ödev 2: Tool-Calling Asistan

## 2.1 Veritabanı Katmanı (`database.py`)

SQLite şeması: `hizmetler`, `araclar`, `siparisler`.
Sevkiyat oluşturulduğunda araç kapasitesinden düşülür; iptal/teslimde geri eklenir.


In [8]:
database_kodu = r'''"""
Veritabanı katmanı — SQLite
============================
Kargo sevkiyat sistemi için şema ve CRUD işlemleri.

Tablolar:
    hizmetler  — kargo hizmet tipleri (ekspres, standart, palet vb.)
    siparisler — oluşturulan sevkiyat kayıtları
    araclar    — filo kapasitesi (stok/kapasite düşme senaryosu)
"""

import sqlite3
import random
import string
from datetime import datetime, timedelta
from pathlib import Path

DB_PATH = Path(__file__).parent / "lojistik.db"


def baglanti():
    """Satırları dict gibi okunabilir şekilde döndüren bağlantı."""
    conn = sqlite3.connect(DB_PATH, check_same_thread=False)
    conn.row_factory = sqlite3.Row
    return conn


# ----------------------------------------------------------------------
# Şema ve başlangıç verisi
# ----------------------------------------------------------------------

SEMA = """
CREATE TABLE IF NOT EXISTS hizmetler (
    kod            TEXT PRIMARY KEY,
    ad             TEXT NOT NULL,
    kategori       TEXT NOT NULL,
    birim_fiyat    REAL NOT NULL,
    max_agirlik_kg REAL NOT NULL,
    teslim_gun     INTEGER NOT NULL,
    aktif          INTEGER DEFAULT 1
);

CREATE TABLE IF NOT EXISTS araclar (
    plaka          TEXT PRIMARY KEY,
    tip            TEXT NOT NULL,
    kapasite_kg    REAL NOT NULL,
    dolu_kg        REAL DEFAULT 0,
    sehir          TEXT NOT NULL
);

CREATE TABLE IF NOT EXISTS siparisler (
    takip_no       TEXT PRIMARY KEY,
    hizmet_kod     TEXT NOT NULL,
    gonderici      TEXT NOT NULL,
    alici          TEXT NOT NULL,
    cikis_sehir    TEXT NOT NULL,
    varis_sehir    TEXT NOT NULL,
    agirlik_kg     REAL NOT NULL,
    tutar          REAL NOT NULL,
    durum          TEXT DEFAULT 'hazirlaniyor',
    olusturma      TEXT NOT NULL,
    tahmini_teslim TEXT NOT NULL,
    arac_plaka     TEXT,
    FOREIGN KEY (hizmet_kod) REFERENCES hizmetler(kod),
    FOREIGN KEY (arac_plaka) REFERENCES araclar(plaka)
);
"""

BASLANGIC_HIZMETLER = [
    ("EKS", "Ekspres Kargo",      "hizli",     18.5, 30,    1, 1),
    ("STD", "Standart Kargo",     "ekonomik",   9.0, 30,    3, 1),
    ("EKO", "Ekonomik Kargo",     "ekonomik",   6.5, 50,    5, 1),
    ("PLT", "Palet Taşıma",       "agir",       4.2, 1200,  4, 1),
    ("SGK", "Soğuk Zincir",       "ozel",      28.0, 200,   2, 1),
    ("KRM", "Kırılabilir Eşya",   "ozel",      22.0, 40,    3, 1),
    ("ULS", "Uluslararası Kargo", "uluslararasi", 45.0, 100, 7, 1),
]

BASLANGIC_ARACLAR = [
    ("34 LJ 1001", "kamyonet", 1500, 0, "İstanbul"),
    ("34 LJ 1002", "kamyonet", 1500, 320, "İstanbul"),
    ("06 LJ 2001", "kamyon",   8000, 1200, "Ankara"),
    ("35 LJ 3001", "kamyonet", 1500, 0, "İzmir"),
    ("33 LJ 4001", "tir",     22000, 8400, "Mersin"),
]


def veritabani_kur(sifirla: bool = False):
    """Şemayı oluşturur ve başlangıç verisini yükler."""
    if sifirla and DB_PATH.exists():
        DB_PATH.unlink()

    conn = baglanti()
    conn.executescript(SEMA)

    # Hizmetler
    mevcut = conn.execute("SELECT COUNT(*) c FROM hizmetler").fetchone()["c"]
    if mevcut == 0:
        conn.executemany(
            "INSERT INTO hizmetler VALUES (?,?,?,?,?,?,?)", BASLANGIC_HIZMETLER
        )

    # Araçlar
    mevcut = conn.execute("SELECT COUNT(*) c FROM araclar").fetchone()["c"]
    if mevcut == 0:
        conn.executemany("INSERT INTO araclar VALUES (?,?,?,?,?)", BASLANGIC_ARACLAR)

    # Örnek sipariş (durum sorgusu demosu için)
    mevcut = conn.execute("SELECT COUNT(*) c FROM siparisler").fetchone()["c"]
    if mevcut == 0:
        conn.execute(
            """INSERT INTO siparisler VALUES
               ('LJ2607DEMO','STD','Anadolu Tekstil','Marmara Market',
                'İstanbul','Ankara',12.5,112.5,'yolda',
                ?,?,'06 LJ 2001')""",
            (
                (datetime.now() - timedelta(days=1)).strftime("%Y-%m-%d %H:%M"),
                (datetime.now() + timedelta(days=2)).strftime("%Y-%m-%d"),
            ),
        )

    conn.commit()
    conn.close()


def takip_no_uret() -> str:
    """LJ + yıl/ay + 4 karakter benzersiz takip numarası."""
    ek = "".join(random.choices(string.ascii_uppercase + string.digits, k=4))
    return f"LJ{datetime.now():%y%m}{ek}"


# ----------------------------------------------------------------------
# Okuma işlemleri
# ----------------------------------------------------------------------

def hizmetleri_listele(kategori: str = None) -> list[dict]:
    conn = baglanti()
    if kategori:
        rows = conn.execute(
            "SELECT * FROM hizmetler WHERE aktif=1 AND kategori=? ORDER BY birim_fiyat",
            (kategori.lower(),),
        ).fetchall()
    else:
        rows = conn.execute(
            "SELECT * FROM hizmetler WHERE aktif=1 ORDER BY birim_fiyat"
        ).fetchall()
    conn.close()
    return [dict(r) for r in rows]


def hizmet_getir(kod: str) -> dict | None:
    conn = baglanti()
    row = conn.execute(
        "SELECT * FROM hizmetler WHERE kod=? AND aktif=1", (kod.upper(),)
    ).fetchone()
    conn.close()
    return dict(row) if row else None


def siparis_getir(takip_no: str) -> dict | None:
    conn = baglanti()
    row = conn.execute(
        """SELECT s.*, h.ad hizmet_ad, h.teslim_gun
           FROM siparisler s JOIN hizmetler h ON s.hizmet_kod = h.kod
           WHERE s.takip_no = ?""",
        (takip_no.upper(),),
    ).fetchone()
    conn.close()
    return dict(row) if row else None


def musait_arac_bul(sehir: str, agirlik_kg: float) -> dict | None:
    """Şehirde yeterli boş kapasitesi olan aracı döndürür."""
    conn = baglanti()
    row = conn.execute(
        """SELECT * FROM araclar
           WHERE sehir = ? AND (kapasite_kg - dolu_kg) >= ?
           ORDER BY (kapasite_kg - dolu_kg) ASC LIMIT 1""",
        (sehir, agirlik_kg),
    ).fetchone()
    conn.close()
    return dict(row) if row else None


# ----------------------------------------------------------------------
# Yazma işlemleri
# ----------------------------------------------------------------------

def siparis_olustur(
    hizmet_kod: str,
    gonderici: str,
    alici: str,
    cikis_sehir: str,
    varis_sehir: str,
    agirlik_kg: float,
) -> dict:
    """Sipariş kaydeder ve araç kapasitesinden düşer."""
    hizmet = hizmet_getir(hizmet_kod)
    if not hizmet:
        return {"hata": f"'{hizmet_kod}' kodlu hizmet bulunamadı"}

    if agirlik_kg > hizmet["max_agirlik_kg"]:
        return {
            "hata": f"{hizmet['ad']} için maksimum ağırlık "
                    f"{hizmet['max_agirlik_kg']} kg, siz {agirlik_kg} kg girdiniz"
        }

    arac = musait_arac_bul(cikis_sehir, agirlik_kg)
    if not arac:
        return {
            "hata": f"{cikis_sehir} şehrinde {agirlik_kg} kg için müsait araç yok"
        }

    takip_no = takip_no_uret()
    tutar = round(agirlik_kg * hizmet["birim_fiyat"], 2)
    simdi = datetime.now()
    teslim = (simdi + timedelta(days=hizmet["teslim_gun"])).strftime("%Y-%m-%d")

    conn = baglanti()
    conn.execute(
        """INSERT INTO siparisler
           (takip_no, hizmet_kod, gonderici, alici, cikis_sehir, varis_sehir,
            agirlik_kg, tutar, durum, olusturma, tahmini_teslim, arac_plaka)
           VALUES (?,?,?,?,?,?,?,?,'hazirlaniyor',?,?,?)""",
        (takip_no, hizmet["kod"], gonderici, alici, cikis_sehir, varis_sehir,
         agirlik_kg, tutar, simdi.strftime("%Y-%m-%d %H:%M"), teslim, arac["plaka"]),
    )
    # Araç kapasitesini düş
    conn.execute(
        "UPDATE araclar SET dolu_kg = dolu_kg + ? WHERE plaka = ?",
        (agirlik_kg, arac["plaka"]),
    )
    conn.commit()
    conn.close()

    return {
        "takip_no": takip_no,
        "hizmet": hizmet["ad"],
        "gonderici": gonderici,
        "alici": alici,
        "guzergah": f"{cikis_sehir} → {varis_sehir}",
        "agirlik_kg": agirlik_kg,
        "tutar_tl": tutar,
        "durum": "hazirlaniyor",
        "tahmini_teslim": teslim,
        "atanan_arac": arac["plaka"],
    }


def siparis_durum_guncelle(takip_no: str, yeni_durum: str) -> dict:
    """Sipariş durumunu günceller."""
    gecerli = ["hazirlaniyor", "yolda", "dagitimda", "teslim_edildi", "iptal"]
    if yeni_durum not in gecerli:
        return {"hata": f"Geçersiz durum. Seçenekler: {', '.join(gecerli)}"}

    siparis = siparis_getir(takip_no)
    if not siparis:
        return {"hata": f"'{takip_no}' takip numaralı sipariş bulunamadı"}

    conn = baglanti()
    conn.execute(
        "UPDATE siparisler SET durum=? WHERE takip_no=?",
        (yeni_durum, takip_no.upper()),
    )
    # İptal/teslim durumunda araç kapasitesini serbest bırak
    if yeni_durum in ("iptal", "teslim_edildi") and siparis["arac_plaka"]:
        conn.execute(
            "UPDATE araclar SET dolu_kg = MAX(0, dolu_kg - ?) WHERE plaka = ?",
            (siparis["agirlik_kg"], siparis["arac_plaka"]),
        )
    conn.commit()
    conn.close()

    return {
        "takip_no": takip_no.upper(),
        "onceki_durum": siparis["durum"],
        "yeni_durum": yeni_durum,
    }


# Modül yüklendiğinde veritabanını hazırla
veritabani_kur()
'''

with open("database.py", "w", encoding="utf-8") as f:
    f.write(database_kodu)

print(f"✓ database.py yazıldı ({len(database_kodu):,} karakter)")

✓ database.py yazıldı (9,059 karakter)


In [9]:
import importlib, sys
if "database" in sys.modules:
    importlib.reload(sys.modules["database"])
import database as db

db.veritabani_kur(sifirla=True)

conn = db.baglanti()
print("HİZMETLER")
for r in conn.execute("SELECT kod, ad, birim_fiyat, max_agirlik_kg FROM hizmetler"):
    print(f"  {r['kod']}: {r['ad']:<20} {r['birim_fiyat']:>6} TL/kg  max {r['max_agirlik_kg']:>6.0f} kg")

print("\nARAÇLAR")
for r in conn.execute("SELECT * FROM araclar"):
    print(f"  {r['plaka']}  {r['sehir']:<10} {r['dolu_kg']:>7.0f}/{r['kapasite_kg']:<7.0f} kg")
conn.close()

HİZMETLER
  EKS: Ekspres Kargo          18.5 TL/kg  max     30 kg
  STD: Standart Kargo          9.0 TL/kg  max     30 kg
  EKO: Ekonomik Kargo          6.5 TL/kg  max     50 kg
  PLT: Palet Taşıma            4.2 TL/kg  max   1200 kg
  SGK: Soğuk Zincir           28.0 TL/kg  max    200 kg
  KRM: Kırılabilir Eşya       22.0 TL/kg  max     40 kg
  ULS: Uluslararası Kargo     45.0 TL/kg  max    100 kg

ARAÇLAR
  34 LJ 1001  İstanbul         0/1500    kg
  34 LJ 1002  İstanbul       320/1500    kg
  06 LJ 2001  Ankara        1200/8000    kg
  35 LJ 3001  İzmir            0/1500    kg
  33 LJ 4001  Mersin        8400/22000   kg


## 2.2 Araç Katmanı (`tools.py`)

Dört araç: ikisi okuma, ikisi yazma.


In [10]:
tools_kodu = r'''"""
Araç katmanı — Tool / Function tanımları
========================================
Her araç veritabanı katmanını çağırır ve modele JSON döndürür.
Halüsinasyon engelleme: araçlar yalnızca DB'de gerçekten var olan
veriyi döndürür; bulunamayan kayıtlar için açıkça hata mesajı verir.
"""

import database as db


# ======================================================================
# ARAÇ FONKSİYONLARI
# ======================================================================

def get_services(kategori: str = None) -> dict:
    """Kargo hizmetlerini listeler. Opsiyonel kategori filtresi."""
    hizmetler = db.hizmetleri_listele(kategori)

    if not hizmetler:
        mevcut = sorted({h["kategori"] for h in db.hizmetleri_listele()})
        return {
            "hata": f"'{kategori}' kategorisinde hizmet bulunamadı",
            "mevcut_kategoriler": mevcut,
        }

    return {
        "hizmet_sayisi": len(hizmetler),
        "hizmetler": [
            {
                "kod": h["kod"],
                "ad": h["ad"],
                "kategori": h["kategori"],
                "birim_fiyat_tl_kg": h["birim_fiyat"],
                "max_agirlik_kg": h["max_agirlik_kg"],
                "teslim_suresi_gun": h["teslim_gun"],
            }
            for h in hizmetler
        ],
    }


def create_shipment(
    hizmet_kod: str,
    gonderici: str,
    alici: str,
    cikis_sehir: str,
    varis_sehir: str,
    agirlik_kg: float,
) -> dict:
    """Yeni sevkiyat kaydı oluşturur ve araç kapasitesinden düşer."""
    return db.siparis_olustur(
        hizmet_kod=hizmet_kod,
        gonderici=gonderici,
        alici=alici,
        cikis_sehir=cikis_sehir,
        varis_sehir=varis_sehir,
        agirlik_kg=float(agirlik_kg),
    )


def track_shipment(takip_no: str) -> dict:
    """Takip numarasıyla sevkiyat durumunu sorgular."""
    siparis = db.siparis_getir(takip_no)

    if not siparis:
        return {"hata": f"'{takip_no}' takip numaralı sevkiyat bulunamadı"}

    return {
        "takip_no": siparis["takip_no"],
        "hizmet": siparis["hizmet_ad"],
        "gonderici": siparis["gonderici"],
        "alici": siparis["alici"],
        "guzergah": f"{siparis['cikis_sehir']} → {siparis['varis_sehir']}",
        "agirlik_kg": siparis["agirlik_kg"],
        "tutar_tl": siparis["tutar"],
        "durum": siparis["durum"],
        "olusturma": siparis["olusturma"],
        "tahmini_teslim": siparis["tahmini_teslim"],
        "arac": siparis["arac_plaka"],
    }


def update_shipment_status(takip_no: str, yeni_durum: str) -> dict:
    """Sevkiyat durumunu günceller (hazirlaniyor/yolda/dagitimda/teslim_edildi/iptal)."""
    return db.siparis_durum_guncelle(takip_no, yeni_durum)


# ======================================================================
# TOOL TANIMLARI (JSON Schema)
# ======================================================================

TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "get_services",
            "description": (
                "Mevcut kargo hizmetlerini, fiyatlarını ve teslim sürelerini listeler. "
                "Kullanıcı fiyat sorduğunda veya hangi hizmetlerin olduğunu merak "
                "ettiğinde bu aracı çağır. Fiyat tahmininde bulunma, her zaman bu aracı kullan."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "kategori": {
                        "type": "string",
                        "description": (
                            "Filtrelenecek kategori: 'hizli', 'ekonomik', 'agir', "
                            "'ozel', 'uluslararasi'. Boş bırakılırsa tümü listelenir."
                        ),
                        "enum": ["hizli", "ekonomik", "agir", "ozel", "uluslararasi"],
                    }
                },
                "required": [],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "create_shipment",
            "description": (
                "Yeni bir sevkiyat kaydı oluşturur, takip numarası üretir ve "
                "uygun araca kapasite ataması yapar. Tüm parametreler kullanıcıdan "
                "alınmalıdır — eksik bilgi varsa önce kullanıcıya sor, uydurma."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "hizmet_kod": {
                        "type": "string",
                        "description": "Hizmet kodu (EKS, STD, EKO, PLT, SGK, KRM, ULS)",
                    },
                    "gonderici": {"type": "string", "description": "Gönderici adı/firması"},
                    "alici": {"type": "string", "description": "Alıcı adı/firması"},
                    "cikis_sehir": {"type": "string", "description": "Çıkış şehri"},
                    "varis_sehir": {"type": "string", "description": "Varış şehri"},
                    "agirlik_kg": {"type": "number", "description": "Gönderi ağırlığı (kg)"},
                },
                "required": [
                    "hizmet_kod", "gonderici", "alici",
                    "cikis_sehir", "varis_sehir", "agirlik_kg",
                ],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "track_shipment",
            "description": (
                "Takip numarasıyla sevkiyatın güncel durumunu, güzergahını ve "
                "tahmini teslim tarihini sorgular. Numara bulunamazsa bunu açıkça belirt."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "takip_no": {
                        "type": "string",
                        "description": "Takip numarası, örn: 'LJ2607DEMO'",
                    }
                },
                "required": ["takip_no"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "update_shipment_status",
            "description": (
                "Sevkiyatın durumunu günceller. İptal veya teslim durumunda "
                "araç kapasitesi otomatik serbest bırakılır."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "takip_no": {"type": "string", "description": "Takip numarası"},
                    "yeni_durum": {
                        "type": "string",
                        "description": "Yeni durum",
                        "enum": [
                            "hazirlaniyor", "yolda", "dagitimda",
                            "teslim_edildi", "iptal",
                        ],
                    },
                },
                "required": ["takip_no", "yeni_durum"],
            },
        },
    },
]


# İsim → fonksiyon eşlemesi
FONKSIYONLAR = {
    "get_services": get_services,
    "create_shipment": create_shipment,
    "track_shipment": track_shipment,
    "update_shipment_status": update_shipment_status,
}
'''

with open("tools.py", "w", encoding="utf-8") as f:
    f.write(tools_kodu)

print(f"✓ tools.py yazıldı ({len(tools_kodu):,} karakter)")

✓ tools.py yazıldı (7,047 karakter)


In [11]:
if "tools" in sys.modules:
    importlib.reload(sys.modules["tools"])
import tools as T

print(f"✓ {len(T.TOOLS)} araç tanımlı:\n")
for t in T.TOOLS:
    fn = t["function"]
    req = ", ".join(fn["parameters"].get("required", [])) or "—"
    print(f"  {fn['name']}({req})")
    print(f"    {fn['description'][:80]}...\n")

✓ 4 araç tanımlı:

  get_services(—)
    Mevcut kargo hizmetlerini, fiyatlarını ve teslim sürelerini listeler. Kullanıcı ...

  create_shipment(hizmet_kod, gonderici, alici, cikis_sehir, varis_sehir, agirlik_kg)
    Yeni bir sevkiyat kaydı oluşturur, takip numarası üretir ve uygun araca kapasite...

  track_shipment(takip_no)
    Takip numarasıyla sevkiyatın güncel durumunu, güzergahını ve tahmini teslim tari...

  update_shipment_status(takip_no, yeni_durum)
    Sevkiyatın durumunu günceller. İptal veya teslim durumunda araç kapasitesi otoma...



### 2.3 Araçları Tek Tek Test Et

In [12]:
print("=== get_services (tümü) ===")
r = T.get_services()
for h in r["hizmetler"]:
    print(f"  {h['kod']}: {h['ad']:<20} {h['birim_fiyat_tl_kg']:>6} TL/kg")

print("\n=== get_services (kategori filtresi) ===")
print(f"  ozel      → {[h['kod'] for h in T.get_services('ozel')['hizmetler']]}")
print(f"  olmayan   → {T.get_services('uzay')}")

=== get_services (tümü) ===
  PLT: Palet Taşıma            4.2 TL/kg
  EKO: Ekonomik Kargo          6.5 TL/kg
  STD: Standart Kargo          9.0 TL/kg
  EKS: Ekspres Kargo          18.5 TL/kg
  KRM: Kırılabilir Eşya       22.0 TL/kg
  SGK: Soğuk Zincir           28.0 TL/kg
  ULS: Uluslararası Kargo     45.0 TL/kg

=== get_services (kategori filtresi) ===
  ozel      → ['KRM', 'SGK']
  olmayan   → {'hata': "'uzay' kategorisinde hizmet bulunamadı", 'mevcut_kategoriler': ['agir', 'ekonomik', 'hizli', 'ozel', 'uluslararasi']}


In [13]:
print("=== create_shipment (başarılı) ===")
r = T.create_shipment("EKS", "Ege Tekstil", "Anadolu Market",
                      "İstanbul", "Ankara", 15.5)
print(json.dumps(r, ensure_ascii=False, indent=2))
yeni_takip = r["takip_no"]

print("\n=== Araç kapasitesi düştü mü? ===")
conn = db.baglanti()
for x in conn.execute("SELECT plaka, dolu_kg, kapasite_kg FROM araclar WHERE sehir='İstanbul'"):
    print(f"  {x['plaka']}: {x['dolu_kg']}/{x['kapasite_kg']} kg")
conn.close()

=== create_shipment (başarılı) ===
{
  "takip_no": "LJ2608VIJA",
  "hizmet": "Ekspres Kargo",
  "gonderici": "Ege Tekstil",
  "alici": "Anadolu Market",
  "guzergah": "İstanbul → Ankara",
  "agirlik_kg": 15.5,
  "tutar_tl": 286.75,
  "durum": "hazirlaniyor",
  "tahmini_teslim": "2026-08-05",
  "atanan_arac": "34 LJ 1002"
}

=== Araç kapasitesi düştü mü? ===
  34 LJ 1001: 0.0/1500.0 kg
  34 LJ 1002: 335.5/1500.0 kg


In [14]:
print("=== Halüsinasyon engelleme testleri ===\n")

print("1) Olmayan takip numarası:")
print(f"   {T.track_shipment('SAHTE999')}\n")

print("2) Ağırlık limiti aşımı:")
print(f"   {T.create_shipment('EKS','A','B','İstanbul','Ankara',500)}\n")

print("3) Olmayan hizmet kodu:")
print(f"   {T.create_shipment('XYZ','A','B','İstanbul','Ankara',10)}\n")

print("4) Müsait araç olmayan şehir:")
print(f"   {T.create_shipment('PLT','A','B','Erzurum','Ankara',900)}\n")

print("5) Geçersiz durum:")
print(f"   {T.update_shipment_status(yeni_takip, 'uçuyor')}")

=== Halüsinasyon engelleme testleri ===

1) Olmayan takip numarası:
   {'hata': "'SAHTE999' takip numaralı sevkiyat bulunamadı"}

2) Ağırlık limiti aşımı:
   {'hata': 'Ekspres Kargo için maksimum ağırlık 30.0 kg, siz 500.0 kg girdiniz'}

3) Olmayan hizmet kodu:
   {'hata': "'XYZ' kodlu hizmet bulunamadı"}

4) Müsait araç olmayan şehir:
   {'hata': 'Erzurum şehrinde 900.0 kg için müsait araç yok'}

5) Geçersiz durum:
   {'hata': 'Geçersiz durum. Seçenekler: hazirlaniyor, yolda, dagitimda, teslim_edildi, iptal'}


In [15]:
print("=== Durum güncelleme + kapasite iadesi ===")
print(f"  yolda      → {T.update_shipment_status(yeni_takip, 'yolda')}")
print(f"  iptal      → {T.update_shipment_status(yeni_takip, 'iptal')}")

conn = db.baglanti()
print("\nKapasite serbest bırakıldı mı?")
for x in conn.execute("SELECT plaka, dolu_kg FROM araclar WHERE sehir='İstanbul'"):
    print(f"  {x['plaka']}: {x['dolu_kg']} kg")
conn.close()

=== Durum güncelleme + kapasite iadesi ===
  yolda      → {'takip_no': 'LJ2608VIJA', 'onceki_durum': 'hazirlaniyor', 'yeni_durum': 'yolda'}
  iptal      → {'takip_no': 'LJ2608VIJA', 'onceki_durum': 'yolda', 'yeni_durum': 'iptal'}

Kapasite serbest bırakıldı mı?
  34 LJ 1001: 0.0 kg
  34 LJ 1002: 320.0 kg


## 2.4 Ajan Katmanı (`agent.py`)

Tool calling döngüsü + sistem promptu + chat template entegrasyonu.


In [16]:
agent_kodu = r'''"""
Ajan katmanı — Tool calling döngüsü ve prompt yönetimi
======================================================
Model ile araçlar arasındaki çok turlu döngüyü yönetir.
Chat template (chat_template.jinja) ile prompt formatını gösterir.
"""

import os
import json
from pathlib import Path

from openai import OpenAI
from jinja2 import Environment, BaseLoader

from tools import TOOLS, FONKSIYONLAR

MODEL = os.environ.get("LOJISTIK_MODEL", "gpt-4o-mini")
MAX_TURNS = 6

_client = None


def client() -> OpenAI:
    """OpenAI istemcisini tembel (lazy) başlatır."""
    global _client
    if _client is None:
        _client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))
    return _client


# ----------------------------------------------------------------------
# Sistem promptu — halüsinasyon engelleme kuralları
# ----------------------------------------------------------------------

SISTEM_PROMPTU = """Sen LojistikAI'sın — Cihat Yıldız tarafından geliştirilen bir kargo ve
sevkiyat operasyon asistanısın. Kullanıcıların hizmet sorgulaması, sevkiyat oluşturması ve
gönderi takibi yapmasına yardımcı olursun.

KESİN KURALLAR:
1. Fiyat, hizmet, takip numarası veya sevkiyat bilgisi verirken MUTLAKA ilgili aracı çağır.
   Hiçbir koşulda kendi bilginle fiyat veya hizmet uydurma.
2. Araç bir kayıt bulamazsa ("hata" alanı dönerse), bunu kullanıcıya açıkça söyle.
   Var gibi davranma, alternatif uydurma.
3. Sevkiyat oluşturmak için gereken bilgilerden biri eksikse ÖNCE kullanıcıya sor.
   Eksik parametreyi kendin doldurma.
4. Hizmet kodunu kullanıcı bilmiyorsa önce get_services ile listeyi göster.
5. Yanıtlarını Türkçe, kısa ve net yaz. Tutarları ve takip numaralarını
   araçtan geldiği şekilde aktar.

Mevcut hizmet kodları get_services aracından öğrenilir — ezberden kod önerme."""


# ----------------------------------------------------------------------
# Chat template (Jinja2) — prompt formatını görselleştirmek için
# ----------------------------------------------------------------------

def _template_yukle():
    yol = Path(__file__).parent / "chat_template.jinja"
    if not yol.exists():
        return None

    def raise_exception(msg):
        raise ValueError(msg)

    env = Environment(loader=BaseLoader())
    env.globals["raise_exception"] = raise_exception
    env.policies["json.dumps_kwargs"] = {"ensure_ascii": False}
    return env.from_string(yol.read_text(encoding="utf-8"))


_TEMPLATE = _template_yukle()


def prompt_onizle(mesajlar: list, tools=None, add_generation_prompt=True) -> str:
    """Mesaj listesini chat template ile formatlar (görüntüleme amaçlı)."""
    if _TEMPLATE is None:
        return "(chat_template.jinja bulunamadı)"
    try:
        return _TEMPLATE.render(
            messages=mesajlar,
            tools=tools if tools is not None else TOOLS,
            add_generation_prompt=add_generation_prompt,
        )
    except Exception as e:
        return f"(Şablon render hatası: {e})"


# ----------------------------------------------------------------------
# Tool calling döngüsü
# ----------------------------------------------------------------------

def _log_satiri(ad: str, argumanlar: dict, sonuc: dict) -> str:
    arg = ", ".join(f"{k}={v!r}" for k, v in argumanlar.items())
    return f"   -> {ad}({arg})\n   <- {json.dumps(sonuc, ensure_ascii=False)}"


def sohbet(mesaj: str, gecmis: list = None) -> tuple[str, str, str]:
    """
    Kullanıcı mesajını işler.

    Dönüş: (yanit, arac_logu, ham_prompt)
    """
    if not os.environ.get("OPENAI_API_KEY"):
        return "⚠ OPENAI_API_KEY tanımlı değil.", "", ""

    mesajlar = [{"role": "system", "content": SISTEM_PROMPTU}]

    for h in (gecmis or []):
        if isinstance(h, dict) and h.get("role") in ("user", "assistant"):
            mesajlar.append({"role": h["role"], "content": h["content"]})

    mesajlar.append({"role": "user", "content": mesaj})

    # İlk isteğin prompt görünümü
    ham_prompt = prompt_onizle(mesajlar)

    log = []
    for turn in range(1, MAX_TURNS + 1):
        try:
            yanit = client().chat.completions.create(
                model=MODEL,
                messages=mesajlar,
                tools=TOOLS,
                tool_choice="auto",
                temperature=0.2,
            )
        except Exception as e:
            return f"⚠ Model hatası: {e}", "\n".join(log), ham_prompt

        msg = yanit.choices[0].message

        # Araç çağrısı yoksa → nihai yanıt
        if not msg.tool_calls:
            log.append(f"\n[Turn {turn}] Nihai yanıt üretildi.")
            mesajlar.append({"role": "assistant", "content": msg.content})
            return msg.content, "\n".join(log), prompt_onizle(mesajlar, add_generation_prompt=False)

        log.append(f"\n[Turn {turn}] Araç Çağrıları:")
        mesajlar.append({
            "role": "assistant",
            "content": msg.content,
            "tool_calls": [
                {
                    "id": tc.id,
                    "type": "function",
                    "function": {
                        "name": tc.function.name,
                        "arguments": tc.function.arguments,
                    },
                }
                for tc in msg.tool_calls
            ],
        })

        for tc in msg.tool_calls:
            ad = tc.function.name
            try:
                argumanlar = json.loads(tc.function.arguments)
            except json.JSONDecodeError:
                argumanlar = {}

            fn = FONKSIYONLAR.get(ad)
            if fn is None:
                sonuc = {"hata": f"Bilinmeyen araç: {ad}"}
            else:
                try:
                    sonuc = fn(**argumanlar)
                except TypeError as e:
                    sonuc = {"hata": f"Eksik/hatalı parametre: {e}"}
                except Exception as e:
                    sonuc = {"hata": f"Araç çalıştırılamadı: {e}"}

            log.append(_log_satiri(ad, argumanlar, sonuc))

            mesajlar.append({
                "role": "tool",
                "tool_call_id": tc.id,
                "name": ad,
                "content": json.dumps(sonuc, ensure_ascii=False),
            })

    return (
        "⚠ Maksimum araç turu aşıldı.",
        "\n".join(log),
        prompt_onizle(mesajlar, add_generation_prompt=False),
    )
'''

with open("agent.py", "w", encoding="utf-8") as f:
    f.write(agent_kodu)

print(f"✓ agent.py yazıldı ({len(agent_kodu):,} karakter)")

✓ agent.py yazıldı (6,313 karakter)


In [17]:
if "agent" in sys.modules:
    importlib.reload(sys.modules["agent"])
from agent import sohbet, prompt_onizle, SISTEM_PROMPTU

print("SİSTEM PROMPTU:")
print("-" * 60)
print(SISTEM_PROMPTU)

SİSTEM PROMPTU:
------------------------------------------------------------
Sen LojistikAI'sın — Cihat Yıldız tarafından geliştirilen bir kargo ve
sevkiyat operasyon asistanısın. Kullanıcıların hizmet sorgulaması, sevkiyat oluşturması ve
gönderi takibi yapmasına yardımcı olursun.

KESİN KURALLAR:
1. Fiyat, hizmet, takip numarası veya sevkiyat bilgisi verirken MUTLAKA ilgili aracı çağır.
   Hiçbir koşulda kendi bilginle fiyat veya hizmet uydurma.
2. Araç bir kayıt bulamazsa ("hata" alanı dönerse), bunu kullanıcıya açıkça söyle.
   Var gibi davranma, alternatif uydurma.
3. Sevkiyat oluşturmak için gereken bilgilerden biri eksikse ÖNCE kullanıcıya sor.
   Eksik parametreyi kendin doldurma.
4. Hizmet kodunu kullanıcı bilmiyorsa önce get_services ile listeyi göster.
5. Yanıtlarını Türkçe, kısa ve net yaz. Tutarları ve takip numaralarını
   araçtan geldiği şekilde aktar.

Mevcut hizmet kodları get_services aracından öğrenilir — ezberden kod önerme.


### 2.5 Uçtan Uca Tool Calling Testleri

In [18]:
# Test 1 — hizmet listesi (okuma)
soru = "Hangi kargo hizmetleri var, fiyatları nedir?"
yanit, log, prompt = sohbet(soru, [])

print(f"KULLANICI: {soru}")
print("=" * 70)
print(log)
print("=" * 70)
print(f"YANIT:\n{yanit}")

KULLANICI: Hangi kargo hizmetleri var, fiyatları nedir?

[Turn 1] Araç Çağrıları:
   -> get_services()
   <- {"hizmet_sayisi": 7, "hizmetler": [{"kod": "PLT", "ad": "Palet Taşıma", "kategori": "agir", "birim_fiyat_tl_kg": 4.2, "max_agirlik_kg": 1200.0, "teslim_suresi_gun": 4}, {"kod": "EKO", "ad": "Ekonomik Kargo", "kategori": "ekonomik", "birim_fiyat_tl_kg": 6.5, "max_agirlik_kg": 50.0, "teslim_suresi_gun": 5}, {"kod": "STD", "ad": "Standart Kargo", "kategori": "ekonomik", "birim_fiyat_tl_kg": 9.0, "max_agirlik_kg": 30.0, "teslim_suresi_gun": 3}, {"kod": "EKS", "ad": "Ekspres Kargo", "kategori": "hizli", "birim_fiyat_tl_kg": 18.5, "max_agirlik_kg": 30.0, "teslim_suresi_gun": 1}, {"kod": "KRM", "ad": "Kırılabilir Eşya", "kategori": "ozel", "birim_fiyat_tl_kg": 22.0, "max_agirlik_kg": 40.0, "teslim_suresi_gun": 3}, {"kod": "SGK", "ad": "Soğuk Zincir", "kategori": "ozel", "birim_fiyat_tl_kg": 28.0, "max_agirlik_kg": 200.0, "teslim_suresi_gun": 2}, {"kod": "ULS", "ad": "Uluslararası Kargo

In [19]:
# Test 2 — sevkiyat oluşturma (yazma)
soru = ("İstanbul'dan Ankara'ya 15.5 kg ekspres gönderi oluştur. "
        "Gönderici Ege Tekstil, alıcı Anadolu Market.")
yanit, log, prompt = sohbet(soru, [])

print(f"KULLANICI: {soru}")
print("=" * 70)
print(log)
print("=" * 70)
print(f"YANIT:\n{yanit}")

KULLANICI: İstanbul'dan Ankara'ya 15.5 kg ekspres gönderi oluştur. Gönderici Ege Tekstil, alıcı Anadolu Market.

[Turn 1] Nihai yanıt üretildi.
YANIT:
Hizmet kodunu belirtmediniz. Mevcut hizmetleri görmek için bir an bekleyin.


In [20]:
# Test 3 — HALÜSİNASYON TESTİ (olmayan kayıt)
soru = "SAHTE999 numaralı kargom nerede?"
yanit, log, prompt = sohbet(soru, [])

print(f"KULLANICI: {soru}")
print("=" * 70)
print(log)
print("=" * 70)
print(f"YANIT:\n{yanit}")
print("\n→ Model kayıt uydurmadıysa test başarılı.")

KULLANICI: SAHTE999 numaralı kargom nerede?

[Turn 1] Araç Çağrıları:
   -> track_shipment(takip_no='SAHTE999')
   <- {"hata": "'SAHTE999' takip numaralı sevkiyat bulunamadı"}

[Turn 2] Nihai yanıt üretildi.
YANIT:
'SAHTE999' takip numaralı sevkiyat bulunamadı. Lütfen takip numarasını kontrol edin veya başka bir numara ile tekrar deneyin.

→ Model kayıt uydurmadıysa test başarılı.


In [21]:
# Test 4 — iş kuralı ihlali (ağırlık limiti)
soru = "İstanbul'dan Ankara'ya 500 kg ekspres gönderi oluştur. Gönderici A, alıcı B."
yanit, log, prompt = sohbet(soru, [])

print(log)
print("=" * 70)
print(f"YANIT:\n{yanit}")


[Turn 1] Araç Çağrıları:
   -> get_services()
   <- {"hizmet_sayisi": 7, "hizmetler": [{"kod": "PLT", "ad": "Palet Taşıma", "kategori": "agir", "birim_fiyat_tl_kg": 4.2, "max_agirlik_kg": 1200.0, "teslim_suresi_gun": 4}, {"kod": "EKO", "ad": "Ekonomik Kargo", "kategori": "ekonomik", "birim_fiyat_tl_kg": 6.5, "max_agirlik_kg": 50.0, "teslim_suresi_gun": 5}, {"kod": "STD", "ad": "Standart Kargo", "kategori": "ekonomik", "birim_fiyat_tl_kg": 9.0, "max_agirlik_kg": 30.0, "teslim_suresi_gun": 3}, {"kod": "EKS", "ad": "Ekspres Kargo", "kategori": "hizli", "birim_fiyat_tl_kg": 18.5, "max_agirlik_kg": 30.0, "teslim_suresi_gun": 1}, {"kod": "KRM", "ad": "Kırılabilir Eşya", "kategori": "ozel", "birim_fiyat_tl_kg": 22.0, "max_agirlik_kg": 40.0, "teslim_suresi_gun": 3}, {"kod": "SGK", "ad": "Soğuk Zincir", "kategori": "ozel", "birim_fiyat_tl_kg": 28.0, "max_agirlik_kg": 200.0, "teslim_suresi_gun": 2}, {"kod": "ULS", "ad": "Uluslararası Kargo", "kategori": "uluslararasi", "birim_fiyat_tl_kg": 45.0

In [22]:
# Test 5 — chat template ile formatlanmış prompt
print("CHAT TEMPLATE ÇIKTISI (son sohbetin promptu)")
print("=" * 70)
print(prompt[:1500])
print("\n   ... (kısaltıldı) ...")

CHAT TEMPLATE ÇIKTISI (son sohbetin promptu)
<|im_start|>system
Sen LojistikAI'sın — Cihat Yıldız tarafından geliştirilen bir kargo ve
sevkiyat operasyon asistanısın. Kullanıcıların hizmet sorgulaması, sevkiyat oluşturması ve
gönderi takibi yapmasına yardımcı olursun.

KESİN KURALLAR:
1. Fiyat, hizmet, takip numarası veya sevkiyat bilgisi verirken MUTLAKA ilgili aracı çağır.
   Hiçbir koşulda kendi bilginle fiyat veya hizmet uydurma.
2. Araç bir kayıt bulamazsa ("hata" alanı dönerse), bunu kullanıcıya açıkça söyle.
   Var gibi davranma, alternatif uydurma.
3. Sevkiyat oluşturmak için gereken bilgilerden biri eksikse ÖNCE kullanıcıya sor.
   Eksik parametreyi kendin doldurma.
4. Hizmet kodunu kullanıcı bilmiyorsa önce get_services ile listeyi göster.
5. Yanıtlarını Türkçe, kısa ve net yaz. Tutarları ve takip numaralarını
   araçtan geldiği şekilde aktar.

Mevcut hizmet kodları get_services aracından öğrenilir — ezberden kod önerme.

# Kullanılabilir Araçlar

Aşağıdaki araçları kullanabi

## 2.6 Gradio Arayüzü

Üç sekmeli panel: tool call logu, chat template önizlemesi, canlı veritabanı durumu.


In [23]:
app_kodu = r'''"""
LojistikAI — Kargo Sevkiyat Asistanı
=====================================
Tool-calling destekli, SQLite veritabanına okuma/yazma yapan asistan.

Katmanlar:
    database.py         → SQLite şeması ve CRUD
    tools.py            → araç fonksiyonları + JSON şemaları
    agent.py            → tool calling döngüsü, prompt yönetimi
    chat_template.jinja → özel Jinja2 chat template
    app.py              → Gradio arayüzü (bu dosya)

Yazar: Cihat Yıldız
"""

import os
import gradio as gr

import database as db
from agent import sohbet, prompt_onizle, SISTEM_PROMPTU
from tools import TOOLS

# ----------------------------------------------------------------------
# ZeroGPU uyumluluğu
# ----------------------------------------------------------------------
# Bu uygulama GPU kullanmaz. Ancak Hugging Face ZeroGPU donanımı Space'in
# başlatılabilmesi için en az bir @spaces.GPU fonksiyonu bulunmasını
# zorunlu kılıyor. Aşağıdaki yer tutucu bu şartı karşılar.
try:
    import spaces

    @spaces.GPU(duration=5)
    def _zerogpu_placeholder():
        return "ok"

except ImportError:
    pass


ORNEK_SORULAR = [
    "Hangi kargo hizmetleri var, fiyatları nedir?",
    "Soğuk zincir ve kırılabilir eşya seçeneklerini göster",
    "İstanbul'dan Ankara'ya 15 kg ekspres gönderi oluştur. Gönderici Ege Tekstil, alıcı Anadolu Market.",
    "LJ2607DEMO takip numaralı gönderim nerede?",
    "SAHTE999 numaralı kargom nerede?",
    "500 kg'lık palet gönderisi İzmir'den Ankara'ya, gönderici Ege Lojistik alıcı Başkent Depo",
]


def _veritabani_ozeti() -> str:
    """Mevcut DB durumunu özet metin olarak döndürür."""
    conn = db.baglanti()
    hizmet_sayisi = conn.execute("SELECT COUNT(*) c FROM hizmetler").fetchone()["c"]
    siparis_sayisi = conn.execute("SELECT COUNT(*) c FROM siparisler").fetchone()["c"]

    satirlar = ["ARAÇ FİLOSU", "-" * 46]
    for r in conn.execute("SELECT * FROM araclar ORDER BY sehir"):
        bos = r["kapasite_kg"] - r["dolu_kg"]
        oran = r["dolu_kg"] / r["kapasite_kg"] * 100
        satirlar.append(
            f"{r['plaka']:<12} {r['sehir']:<10} "
            f"{r['dolu_kg']:>7.0f}/{r['kapasite_kg']:<7.0f} kg  (%{oran:.0f} dolu)"
        )

    satirlar += ["", "SON SEVKİYATLAR", "-" * 46]
    rows = conn.execute(
        "SELECT takip_no, durum, cikis_sehir, varis_sehir, tutar "
        "FROM siparisler ORDER BY olusturma DESC LIMIT 6"
    ).fetchall()
    if rows:
        for r in rows:
            satirlar.append(
                f"{r['takip_no']:<12} {r['cikis_sehir']}→{r['varis_sehir']:<10} "
                f"{r['durum']:<14} {r['tutar']:>8.2f} TL"
            )
    else:
        satirlar.append("(henüz sevkiyat yok)")

    conn.close()
    return (
        f"Hizmet: {hizmet_sayisi}  |  Sevkiyat: {siparis_sayisi}\n\n"
        + "\n".join(satirlar)
    )


with gr.Blocks(title="LojistikAI — Kargo Asistanı") as demo:
    gr.Markdown(
        """
        # 🚚 LojistikAI — Kargo Sevkiyat Asistanı

        SQLite veritabanına **okuma ve yazma** yapan tool-calling asistanı.
        Model, sorunuza göre uygun aracı çağırır; **yanıtlar yalnızca veritabanından
        dönen gerçek veriye dayanır** — olmayan bir kayıt varmış gibi sunulmaz.

        | Araç | İşlev | DB |
        |------|-------|-----|
        | `get_services` | Hizmet ve fiyat listesi | okuma |
        | `create_shipment` | Sevkiyat kaydı + araç kapasitesi düşme | **yazma** |
        | `track_shipment` | Takip numarasıyla durum sorgusu | okuma |
        | `update_shipment_status` | Durum güncelleme, kapasite iadesi | **yazma** |
        """
    )

    with gr.Row():
        with gr.Column(scale=3):
            chatbot = gr.Chatbot(label="Sohbet", height=430, type="messages")
            with gr.Row():
                giris = gr.Textbox(
                    placeholder="Örn: İstanbul'dan Ankara'ya 15 kg ekspres gönderi oluştur",
                    show_label=False,
                    scale=5,
                    container=False,
                )
                gonder = gr.Button("Gönder", variant="primary", scale=1)
            temizle = gr.Button("🗑 Sohbeti temizle", size="sm")

        with gr.Column(scale=2):
            with gr.Tab("🔧 Tool Calls"):
                arac_logu = gr.Code(
                    label="Araç çağrıları",
                    lines=20,
                    interactive=False,
                    value="Henüz araç çağrısı yapılmadı.",
                )
            with gr.Tab("📝 Chat Template"):
                prompt_kutusu = gr.Code(
                    label="chat_template.jinja ile formatlanmış prompt",
                    lines=20,
                    interactive=False,
                    value="Mesaj gönderdiğinizde formatlanmış prompt burada görünür.",
                )
            with gr.Tab("🗄 Veritabanı"):
                db_kutusu = gr.Code(
                    label="Canlı DB durumu",
                    lines=20,
                    interactive=False,
                    value=_veritabani_ozeti(),
                )
                yenile = gr.Button("🔄 Yenile", size="sm")

    gr.Examples(
        examples=[[s] for s in ORNEK_SORULAR],
        inputs=giris,
        label="Örnek sorular (son ikisi halüsinasyon ve kapasite kontrolünü test eder)",
    )

    def _yanitla(mesaj, gecmis):
        if not mesaj or not mesaj.strip():
            return gecmis, "", "Boş mesaj.", "", _veritabani_ozeti()

        yanit, log, ham_prompt = sohbet(mesaj, gecmis or [])
        yeni_gecmis = (gecmis or []) + [
            {"role": "user", "content": mesaj},
            {"role": "assistant", "content": yanit},
        ]
        return (
            yeni_gecmis,
            "",
            log.strip() or "Bu soruda araç çağrısı yapılmadı.",
            ham_prompt,
            _veritabani_ozeti(),
        )

    ciktilar = [chatbot, giris, arac_logu, prompt_kutusu, db_kutusu]
    gonder.click(_yanitla, [giris, chatbot], ciktilar)
    giris.submit(_yanitla, [giris, chatbot], ciktilar)
    temizle.click(
        lambda: ([], "", "Henüz araç çağrısı yapılmadı.", "", _veritabani_ozeti()),
        None,
        ciktilar,
    )
    yenile.click(_veritabani_ozeti, None, db_kutusu)

    gr.Markdown(
        """
        ---
        **Geliştirici:** Cihat Yıldız ·
        [GitHub](https://github.com/cihatyldz) ·
        [Hugging Face](https://huggingface.co/cihatyldz)
        """
    )


if __name__ == "__main__":
    demo.launch(ssr_mode=False)
'''

with open("app.py", "w", encoding="utf-8") as f:
    f.write(app_kodu)

import ast
ast.parse(app_kodu)
print(f"✓ app.py yazıldı ({len(app_kodu):,} karakter), syntax geçerli")

✓ app.py yazıldı (6,456 karakter), syntax geçerli


In [24]:
requirements = """gradio>=5.0.0
openai>=1.50.0
jinja2>=3.1.0
spaces>=0.30.0
"""

with open("requirements.txt", "w", encoding="utf-8") as f:
    f.write(requirements)

print("✓ requirements.txt yazıldı")
print(requirements)

✓ requirements.txt yazıldı
gradio>=5.0.0
openai>=1.50.0
jinja2>=3.1.0
spaces>=0.30.0



In [25]:
readme = r'''---
title: LojistikAI Kargo Asistanı
emoji: 🚚
colorFrom: indigo
colorTo: blue
sdk: gradio
sdk_version: 5.49.1
app_file: app.py
pinned: false
license: mit
short_description: Tool-calling destekli kargo sevkiyat asistanı (SQLite + custom Jinja2 template)
---

# 🚚 LojistikAI — Kargo Sevkiyat Asistanı

Tool-calling destekli, **SQLite veritabanına hem okuma hem yazma** yapan kargo operasyon asistanı. Model, kullanıcının isteğine göre uygun fonksiyonu çağırır; yanıtlar tamamen veritabanından dönen gerçek veriye dayanır.

🔗 **Canlı demo:** [huggingface.co/spaces/cihatyldz/lojistik-kargo-asistani](https://huggingface.co/spaces/cihatyldz/lojistik-kargo-asistani)

---

## 📋 Senaryo

Bir kargo firmasının operasyon asistanı. Kullanıcı şunları yapabilir:

- Hizmet listesini ve fiyatları sorgulama (kategoriye göre filtreleme)
- Yeni sevkiyat oluşturma — takip numarası üretilir, uygun araca atanır, **kapasiteden düşülür**
- Takip numarasıyla gönderi durumu sorgulama
- Sevkiyat durumu güncelleme — iptal/teslim durumunda **araç kapasitesi serbest bırakılır**

Sistem, ağırlık limiti aşımı, müsait araç bulunmaması ve geçersiz hizmet kodu gibi durumları veritabanı seviyesinde doğrular.

---

## 🧩 Mimari

```
                     Kullanıcı
                         │
                         ▼
                ┌─────────────────┐
                │     app.py      │  Gradio arayüzü
                │  (3 sekmeli UI) │  sohbet · tool log · prompt · DB
                └────────┬────────┘
                         ▼
                ┌─────────────────┐
                │    agent.py     │  tool calling döngüsü
                │                 │  prompt yönetimi (max 6 tur)
                └────────┬────────┘
                    ┌────┴────┐
                    ▼         ▼
          ┌──────────────┐  ┌──────────────────────┐
          │  tools.py    │  │ chat_template.jinja  │
          │ 4 fonksiyon  │  │ rol sarmalama +      │
          │ JSON Schema  │  │ tool call formatı    │
          └──────┬───────┘  └──────────────────────┘
                 ▼
          ┌──────────────┐
          │ database.py  │  SQLite
          │              │  hizmetler · araclar · siparisler
          └──────────────┘
```

| Dosya | Sorumluluk |
|-------|-----------|
| `chat_template.jinja` | Rol sarmalama (`system`/`user`/`assistant`/`tool`), tool call formatı |
| `database.py` | SQLite şeması, CRUD, kapasite yönetimi, takip no üretimi |
| `tools.py` | Araç fonksiyonları + OpenAI JSON Schema tanımları |
| `agent.py` | Tool calling döngüsü, sistem promptu, template render |
| `app.py` | Gradio arayüzü — sohbet, tool log, prompt önizleme, canlı DB |

---

## 🗄 Veritabanı Şeması

```sql
hizmetler (kod, ad, kategori, birim_fiyat, max_agirlik_kg, teslim_gun, aktif)
araclar   (plaka, tip, kapasite_kg, dolu_kg, sehir)
siparisler(takip_no, hizmet_kod, gonderici, alici, cikis_sehir, varis_sehir,
           agirlik_kg, tutar, durum, olusturma, tahmini_teslim, arac_plaka)
```

Başlangıç verisi: 7 hizmet tipi (EKS, STD, EKO, PLT, SGK, KRM, ULS), 5 araç, 1 demo sevkiyat.

---

## 🔧 Araçlar

| Araç | Açıklama | DB İşlemi |
|------|----------|-----------|
| `get_services(kategori?)` | Hizmet ve fiyat listesi | SELECT |
| `create_shipment(...)` | Sevkiyat kaydı + araç ataması | INSERT + UPDATE |
| `track_shipment(takip_no)` | Durum sorgulama | SELECT (JOIN) |
| `update_shipment_status(takip_no, durum)` | Durum güncelleme | UPDATE (+ kapasite iadesi) |

---

## 📝 Custom Chat Template

`chat_template.jinja` dosyası dört rolü ayrı ayrı sarmalar ve tool calling'i destekler:

```jinja
{%- if message['role'] == 'assistant' -%}
    {{- '<|im_start|>assistant\n' -}}
    {%- if message['tool_calls'] is defined and message['tool_calls'] -%}
        {%- for tc in message['tool_calls'] -%}
            {{- '\n<|tool_call|>' -}}
            {{- '{"name": "' + fn['name'] + '", "arguments": ' -}}
            {{- fn['arguments'] | tojson -}}
            {{- '}<|/tool_call|>' -}}
        {%- endfor -%}
    {%- endif -%}
    {{- '<|im_end|>\n' -}}
{%- endif -%}
```

Üretilen prompt formatı:

```text
<|im_start|>system
Sen LojistikAI'sın — ...

# Kullanılabilir Araçlar
<|tool_call|>{"name": "araç_adı", "arguments": {...}}<|/tool_call|>

Araç tanımları (JSON Schema):
{"name": "get_services", "description": "...", "parameters": {...}}
<|im_end|>
<|im_start|>user
Özel hizmetleri listele<|im_end|>
<|im_start|>assistant
<|tool_call|>{"name": "get_services", "arguments": {"kategori":"ozel"}}<|/tool_call|><|im_end|>
<|im_start|>tool
[get_services] <|tool_response|>{"hizmet_sayisi":2,...}<|/tool_response|><|im_end|>
<|im_start|>assistant
```

Şablon geçersiz rol geldiğinde `raise_exception` ile hata fırlatır:

```
ValueError: Desteklenmeyen rol: hacker. Geçerli roller: system, user, assistant, tool
```

Arayüzdeki **📝 Chat Template** sekmesinde her mesaj için formatlanmış prompt canlı görüntülenir.

---

## 💬 Örnek Akışlar

### 1) Sevkiyat oluşturma — DB'ye yazma

**Kullanıcı:** *"İstanbul'dan Ankara'ya 15.5 kg ekspres gönderi oluştur. Gönderici Ege Tekstil, alıcı Anadolu Market."*

```text
[Turn 1] Araç Çağrıları:
   -> create_shipment(hizmet_kod='EKS', gonderici='Ege Tekstil',
                      alici='Anadolu Market', cikis_sehir='İstanbul',
                      varis_sehir='Ankara', agirlik_kg=15.5)
   <- {"takip_no": "LJ26083XR7", "hizmet": "Ekspres Kargo",
       "guzergah": "İstanbul → Ankara", "agirlik_kg": 15.5,
       "tutar_tl": 286.75, "durum": "hazirlaniyor",
       "tahmini_teslim": "2026-08-05", "atanan_arac": "34 LJ 1002"}

[Turn 2] Nihai yanıt üretildi.
```

Araç kapasitesi otomatik güncellenir:

```text
34 LJ 1002:  320.0 → 335.5 / 1500.0 kg
```

### 2) Halüsinasyon engelleme — olmayan kayıt

**Kullanıcı:** *"SAHTE999 numaralı kargom nerede?"*

```text
[Turn 1] Araç Çağrıları:
   -> track_shipment(takip_no='SAHTE999')
   <- {"hata": "'SAHTE999' takip numaralı sevkiyat bulunamadı"}

[Turn 2] Nihai yanıt üretildi.
```

Model, kayıt uydurmak yerine bulunamadığını bildirir.

### 3) İş kuralı doğrulama — ağırlık limiti

```text
   -> create_shipment(hizmet_kod='EKS', ..., agirlik_kg=500)
   <- {"hata": "Ekspres Kargo için maksimum ağırlık 30.0 kg,
                siz 500.0 kg girdiniz"}
```

### 4) Durum güncelleme — kapasite iadesi

```text
   -> update_shipment_status(takip_no='LJ26083XR7', yeni_durum='iptal')
   <- {"takip_no": "LJ26083XR7", "onceki_durum": "yolda", "yeni_durum": "iptal"}
```

```text
34 LJ 1002:  335.5 → 320.0 kg   (kapasite serbest bırakıldı)
```

---

## 🛡 Halüsinasyon Engelleme

Üç katmanlı koruma:

**Sistem promptu seviyesinde** — model, fiyat/hizmet/takip bilgisi verirken mutlaka araç çağırmak zorunda; eksik parametreyi kendi doldurmak yerine kullanıcıya sorması isteniyor.

**Araç seviyesinde** — bulunamayan kayıtlar `{"hata": "..."}` döndürür, boş liste veya varsayılan değer değil. `get_services` olmayan bir kategori için mevcut kategorileri de listeler.

**Şablon seviyesinde** — sistem mesajının sonuna otomatik olarak *"Yanıtlarını yalnızca araçlardan dönen gerçek veriye dayandır"* kuralı eklenir.

---

## 🚀 Yerelde Çalıştırma

```bash
git clone https://github.com/<kullanici>/lojistik-kargo-asistani.git
cd lojistik-kargo-asistani

pip install -r requirements.txt

export OPENAI_API_KEY="sk-..."
python app.py
```

Arayüz `http://localhost:7860` adresinde açılır. Veritabanı (`lojistik.db`) ilk çalıştırmada otomatik oluşturulur ve başlangıç verisiyle doldurulur.

### Veritabanını sıfırlama

```python
import database as db
db.veritabani_kur(sifirla=True)
```

### Farklı model kullanma

```bash
export LOJISTIK_MODEL="gpt-4o"
python app.py
```

---

## ☁️ Hugging Face Spaces'e Yayınlama

> **Not:** Hugging Face artık Gradio Space'lerin ücretsiz `cpu-basic` üzerinde oluşturulmasına izin vermiyor. Ücretsiz kişisel hesaplar ZeroGPU üzerinde 2 adede kadar Gradio Space barındırabiliyor. Bu proje `zero-a10g` ile yayınlanmıştır; uygulama GPU kullanmaz, `app.py` içindeki `@spaces.GPU` yer tutucusu yalnızca platform şartını karşılar.

```python
from huggingface_hub import create_repo, HfApi

SPACE_ID = "kullanici/lojistik-kargo-asistani"

create_repo(repo_id=SPACE_ID, repo_type="space",
            space_sdk="gradio", space_hardware="zero-a10g")

api = HfApi()
for f in ["app.py", "agent.py", "tools.py", "database.py",
          "chat_template.jinja", "requirements.txt", "README.md"]:
    api.upload_file(path_or_fileobj=f, path_in_repo=f,
                    repo_id=SPACE_ID, repo_type="space")

api.add_space_secret(repo_id=SPACE_ID, key="OPENAI_API_KEY", value="sk-...")
```

---

## ⚙️ Teknik Detaylar

| Konu | Değer |
|------|-------|
| Model | `gpt-4o-mini` (native function calling) |
| Veritabanı | SQLite (`lojistik.db`, otomatik oluşturulur) |
| Tool sayısı | 4 (2 okuma, 2 yazma) |
| Max tool turu | 6 |
| Sıcaklık | 0.2 (araç seçiminde tutarlılık) |
| Şablon motoru | Jinja2 |

---

## 📁 Dosya Yapısı

```
├── app.py                  # Gradio arayüzü
├── agent.py                # Tool calling döngüsü + prompt yönetimi
├── tools.py                # Araç fonksiyonları + JSON şemaları
├── database.py             # SQLite katmanı
├── chat_template.jinja     # Özel Jinja2 chat template
├── requirements.txt
└── README.md
```

---

## 👤 Yazar

**Cihat Yıldız** — Kıdemli Veri Bilimcisi, Lojistik Sektörü
[Hugging Face](https://huggingface.co/cihatyldz)

## 📄 Lisans

MIT
'''

with open("README.md", "w", encoding="utf-8") as f:
    f.write(readme)

print(f"✓ README.md yazıldı ({len(readme):,} karakter)")

✓ README.md yazıldı (9,378 karakter)


### 2.7 Arayüzü Colab'da Dene

In [27]:
# app.py'yi Gradio 5/6 uyumlu hale getir
kod = open("app.py", encoding="utf-8").read()

kod = kod.replace(
    'chatbot = gr.Chatbot(label="Sohbet", height=430, type="messages")',
    '''_chatbot_kwargs = {"label": "Sohbet", "height": 430}
            if int(gr.__version__.split(".")[0]) < 6:
                _chatbot_kwargs["type"] = "messages"
            chatbot = gr.Chatbot(**_chatbot_kwargs)'''
)

with open("app.py", "w", encoding="utf-8") as f:
    f.write(kod)

import ast
ast.parse(kod)
print(f"✓ app.py güncellendi (Gradio {gr.__version__} uyumlu)")

✓ app.py güncellendi (Gradio 6.20.0 uyumlu)


---
## 3. Hugging Face Space'e Yayınla


In [29]:
from huggingface_hub import HfApi, login, create_repo

login(token=HF_TOKEN)
api = HfApi()

try:
    create_repo(
        repo_id=SPACE_ID,
        repo_type="space",
        space_sdk="gradio",
        space_hardware="zero-a10g",   # ücretsiz slot (cpu-basic artık PRO gerektiriyor)
        private=False,
        exist_ok=True,
    )
    print(f"✓ Space hazır: {SPACE_ID}")
except Exception as e:
    print(f"⚠ {e}")

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


✓ Space hazır: cihatyldz/lojistik-kargo-asistani


In [31]:
# short_description'ı kısalt (HF sınırı: 60 karakter)
readme = open("README.md", encoding="utf-8").read()

eski = "short_description: Tool-calling destekli kargo sevkiyat asistanı (SQLite + custom Jinja2 template)"
yeni = "short_description: Tool-calling kargo asistanı (SQLite + Jinja2)"

readme = readme.replace(eski, yeni)

with open("README.md", "w", encoding="utf-8") as f:
    f.write(readme)

# Doğrula
for satir in readme.split("---")[1].strip().split("\n"):
    if satir.startswith("short_description:"):
        uzunluk = len(satir.replace("short_description:", "").strip())
        print(f"✓ short_description: {uzunluk} karakter (sınır: 60)")

✓ short_description: 45 karakter (sınır: 60)


In [32]:
DOSYALAR = [
    "app.py",
    "agent.py",
    "tools.py",
    "database.py",
    "chat_template.jinja",
    "requirements.txt",
    "README.md",
]

for dosya in DOSYALAR:
    api.upload_file(
        path_or_fileobj=dosya,
        path_in_repo=dosya,
        repo_id=SPACE_ID,
        repo_type="space",
    )
    print(f"  ✓ {dosya}")

print(f"\n✓ {len(DOSYALAR)} dosya yüklendi")

No files have been modified since last commit. Skipping to prevent empty commit.
No files have been modified since last commit. Skipping to prevent empty commit.


  ✓ app.py


No files have been modified since last commit. Skipping to prevent empty commit.


  ✓ agent.py
  ✓ tools.py


No files have been modified since last commit. Skipping to prevent empty commit.
No files have been modified since last commit. Skipping to prevent empty commit.


  ✓ database.py
  ✓ chat_template.jinja


No files have been modified since last commit. Skipping to prevent empty commit.


  ✓ requirements.txt
  ✓ README.md

✓ 7 dosya yüklendi


In [33]:
# OPENAI_API_KEY secret ekle
try:
    api.add_space_secret(repo_id=SPACE_ID, key="OPENAI_API_KEY", value=OPENAI_KEY)
    print("✓ OPENAI_API_KEY secret eklendi")
except Exception as e:
    print(f"⚠ {e}")
    print("  Manuel: Space → Settings → Variables and secrets")

print(f"\n{'='*62}")
print(f"  🚀 CANLI DEMO")
print(f"  https://huggingface.co/spaces/{SPACE_ID}")
print(f"{'='*62}")
print("\nİlk derleme 2-4 dakika sürebilir.")

✓ OPENAI_API_KEY secret eklendi

  🚀 CANLI DEMO
  https://huggingface.co/spaces/cihatyldz/lojistik-kargo-asistani

İlk derleme 2-4 dakika sürebilir.


---
## ✅ Teslim Kontrol Listesi

| # | Gereksinim | Durum |
|---|-----------|-------|
| 1 | `chat_template.jinja` — rol tanımları + tool calling | ☐ |
| 2 | Senaryo belirlendi (kargo sevkiyat) | ☐ |
| 3 | Gerçek veritabanı (SQLite) — okuma **ve** yazma | ☐ |
| 4 | Modüler kod mimarisi (5 dosya, ayrı sorumluluklar) | ☐ |
| 5 | Halüsinasyon engelleme (3 katman) | ☐ |
| 6 | 4 fonksiyon, uçtan uca çalışan akış | ☐ |
| 7 | Gradio arayüzü | ☐ |
| 8 | Hugging Face Space yayında | ☐ |
| 9 | README (senaryo, mimari, kurulum, örnek log) | ☐ |

### Ekran görüntüsü için

README'de istenen "tool-call çıktısını gösteren log ekran görüntüsü" için
yukarıdaki **Test 2** veya **Test 3** hücresinin çıktısını ekran görüntüsü alıp
GitHub reposuna ekleyin. Alternatif olarak Space arayüzündeki
**🔧 Tool Calls** sekmesinin ekran görüntüsü de kullanılabilir.
